# Milestone 3 — Model Architecture & End-to-End Pipeline

Local Mac (Apple Silicon / MPS) pipeline. Orchestrates `src/` modules.

**Sections map to M3 submission requirements:**
1. Model architecture selection
2. Architecture justification
3. Baseline model performance
4. Hyperparameter tuning
5. End-to-end setup
6. Architecture diagram


## 0. Environment & smoke checks


In [ ]:
import os, sys
from pathlib import Path

M3_ROOT = Path.cwd()
if (M3_ROOT / "src").exists() is False:
    M3_ROOT = Path.cwd().parent
sys.path.insert(0, str(M3_ROOT))
os.chdir(M3_ROOT)
os.environ["GS_NO_SIGN_REQUEST"] = "YES"

import torch
from src import config
from src.data.gcs import list_bucket_files
from src.data.loaders import load_firms_raster

print("DEVICE:", config.DEVICE)
print("MPS available:", torch.backends.mps.is_available())
print("Channels:", len(config.FEATURE_CHANNEL_NAMES))
print("FIRMS files:", len(list_bucket_files(config.FIRMS_BUCKET, config.FIRMS_PREFIX, suffix=".tif")))
da = load_firms_raster("2024-08-15")
print("FIRMS sample shape:", da.shape)


## 1. Model architecture selection

- **Baseline:** XGBoost on flattened last-day per-pixel features (30 channels).
- **Primary:** ConvLSTM encoder (7-day sequence) + U-Net-style decoder → `(64,64,1)` risk map.
- **Stretch (deferred):** Swin-UNet — too data-hungry for ~500-sample candidate subset.

See `src/models/baseline.py` and `src/models/convlstm_unet.py`.


## 2. Architecture justification

Task is spatiotemporal binary semantic segmentation. ConvLSTM preserves spatial structure across the 7-day recurrence; U-Net recovers full-resolution maps via skips. Loss is BCE+Dice or Focal (never plain accuracy) given confirmed imbalance.

References: Huot et al. (Next Day Wildfire Spread), Gerard et al. (WildfireSpreadTS), Ronneberger et al. (U-Net), Shi et al. (ConvLSTM).

Full write-up: `../Milestone 3 Plan.md` sections 2–3; diagram: `reports/architecture_diagram.md`.


## 5a. End-to-end — lock AOI + build candidate dataset

Run once (or use CLI). Prefer CLI for long builds:
`python scripts/lock_aoi.py` then `python scripts/build_dataset.py`.


In [ ]:
from src.data.aoi import lock_intersection_aoi, load_locked_aoi
from pathlib import Path
import json

aoi_path = config.METADATA_DIR / "aoi_bounds.json"
if not aoi_path.exists():
    lock_intersection_aoi()
else:
    print("AOI already locked:", load_locked_aoi())
    print(json.dumps(json.load(open(aoi_path)), indent=2)[:800])


In [ ]:
# Long-running — prefer: !python scripts/build_dataset.py --max-days 45 --start 2024-07-01 --end 2024-09-30
# Uncomment to run from notebook:
# !python scripts/build_dataset.py --max-days 45 --start 2024-07-01 --end 2024-09-30
print("Use scripts/build_dataset.py to build the candidate dataset from GCS.")


## 3. Baseline model performance


In [ ]:
from src.data.dataset import load_splits
from src.models.baseline import train_xgboost_baseline
import json

splits, norm_stats, meta = load_splits()
print({k: splits[k]["X"].shape for k in splits})
baseline = train_xgboost_baseline(
    splits["train"]["X"], splits["train"]["y"],
    splits["val"]["X"], splits["val"]["y"],
    splits["test"]["X"], splits["test"]["y"],
)
print("Test:", json.dumps(baseline["metrics"]["test"], indent=2))
print("Floor:", json.dumps(baseline["metrics"]["floor_test"], indent=2))


## Primary model — ConvLSTM+U-Net (default config)


In [ ]:
from src.training.train_loop import train_model, PatchDataset, evaluate
from src.training.losses import get_loss
from torch.utils.data import DataLoader

dl = train_model(
    splits["train"]["X"], splits["train"]["y"],
    splits["val"]["X"], splits["val"]["y"],
    epochs=20,
)
device = next(dl["model"].parameters()).device
loader = DataLoader(PatchDataset(splits["test"]["X"], splits["test"]["y"]), batch_size=4)
_, test_metrics, _, _ = evaluate(dl["model"], loader, device, get_loss("bce_dice"))
print(test_metrics)


## 4. Hyperparameter tuning

Random search (not full grid) — proportionate to candidate size and local MPS compute.


In [ ]:
from src.training.tune import random_search

# Uncomment to run (~minutes–hours depending on subset size):
# tune_out = random_search(
#     splits["train"]["X"], splits["train"]["y"],
#     splits["val"]["X"], splits["val"]["y"],
#     splits["test"]["X"], splits["test"]["y"],
#     n_trials=8, epochs=12,
# )
print("Prefer: python scripts/train_models.py --tune")


## 6. Architecture diagram

See [`reports/architecture_diagram.md`](../reports/architecture_diagram.md).

Judgment calls: [`reports/judgment_calls.md`](../reports/judgment_calls.md).
